# Recommender Evaluation
This notebook compares CalCourse's TF-IDF, semantic, and hybrid ranking approaches using manually labeled student profiles and ranking metrics.

In [1]:
import pandas as pd

courses = pd.read_csv(
    "../data/processed/recommendable_courses_fall_2026.csv"
)

courses["course"] = (
    courses["subject"] + " " + courses["course_number"].astype(str)
)

In [2]:
evaluation_profiles = [
    {
        "name": "Machine Learning / Data Science",
        "interests": "machine learning, data science, predictive modeling, statistics",
        "preferred_subjects": ["DATA", "STAT", "COMPSCI", "INDENG"],
        "relevant_courses": [
            "DATA C102",
            "DATA 145",
            "STAT 154",
            "STAT 159",
            "INDENG 142A",
            "COMPSCI 189",
        ],
    },

    {
        "name": "Product Analytics / PM",
        "interests": "product management, experimentation, customer behavior, marketing analytics",
        "preferred_subjects": ["UGBA", "DATA", "INDENG", "ENGIN"],
        "relevant_courses": [
            "UGBA 104",
            "UGBA 160",
            "UGBA 161",
            "UGBA 162",
            "INDENG 142A",
            "ENGIN 183D",
        ],
    },

    {
        "name": "Economics / Public Policy",
        "interests": "economics, public policy, causal inference, inequality, labor markets",
        "preferred_subjects": ["ECON", "PUBPOL", "STAT", "DATA"],
        "relevant_courses": [
            "ECON 130",
            "ECON 140",
            "ECON 141",
            "ECON 151",
            "PUBPOL 141",
            "ECON 121",
        ],
    },

    {
        "name": "Healthcare / Computational Biology",
        "interests": "healthcare, computational biology, biomedical data, machine learning",
        "preferred_subjects": ["CMPBIO", "BIOENG", "DATA", "STAT"],
        "relevant_courses": [
            "CMPBIO 175",
            "BIOENG 140L",
            "STAT 159",
            "DATA 145",
        ],
    },

    {
        "name": "Engineering Systems / Optimization",
        "interests": "optimization, simulation, engineering systems, applied mathematics",
        "preferred_subjects": ["INDENG", "ENGIN", "MATH", "AEROENG", "CIVENG"],
        "relevant_courses": [
            "INDENG 174",
            "MATH 170",
            "AEROENG C144",
            "CHMENG 130",
            "ELENG 66",
        ],
    },

    {
        "name": "Software / Systems",
        "interests": "databases, distributed systems, backend engineering, operating systems",
        "preferred_subjects": ["COMPSCI", "EECS"],
        "relevant_courses": [
            "COMPSCI 162",
            "COMPSCI 169A",
            "COMPSCI 186",
            "COMPSCI 161",
        ],
    },

    {
        "name": "Statistics",
        "interests": "statistical modeling, probability, inference, time series",
        "preferred_subjects": ["STAT", "DATA", "MATH"],
        "relevant_courses": [
            "STAT 133",
            "STAT 134",
            "STAT 135",
            "STAT 151A",
            "STAT 153",
            "STAT 154",
            "STAT 159",
        ],
    },

    {
        "name": "Finance / Quantitative Analysis",
        "interests": "finance, quantitative analysis, risk, forecasting, markets",
        "preferred_subjects": ["ECON", "STAT", "INDENG", "UGBA"],
        "relevant_courses": [
            "ECON 136",
            "STAT 153",
            "INDENG 172",
            "UGBA 103",
        ],
    },

    {
        "name": "Behavioral Science",
        "interests": "decision making, psychology, behavioral economics, human behavior",
        "preferred_subjects": ["PSYCH", "COGSCI", "ECON", "PUBPOL"],
        "relevant_courses": [
            "COGSCI 151",
            "PUBPOL 141",
            "ECON 119",
        ],
    },

    {
        "name": "Design / HCI",
        "interests": "human computer interaction, product design, user research, interface design",
        "preferred_subjects": ["DESINV", "INFO", "COGSCI", "ENGIN"],
        "relevant_courses": [
            "ENGIN 183D",
        ],
    },
]

Relevance labels were expanded using profile-specific criteria rather than model outputs, reducing the risk of penalizing valid recommendations simply because the original label set was too sparse.

In [3]:
course_labels = set(courses["course"])

for profile in evaluation_profiles:
    missing = [
        course
        for course in profile["relevant_courses"]
        if course not in course_labels
    ]

    if missing:
        print(profile["name"], "missing:", missing)

## 2. Ranking Functions
Each student proile is ranked using three approaches: 

- TF-IDF keyword similarity 
- Semantic embedding similarity
- Hybrid ranking that combines semantic relevance with subject preferences

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [5]:
courses["text"] = (
    courses["title"].fillna("") + ". " +
    courses["description"].fillna("")
)

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    courses["text"]
)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

course_embeddings = embedding_model.encode(
    courses["text"].tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/65 [00:00<?, ?it/s]

In [6]:
def rank_tfidf(profile, top_k=10):
    profile_vector = tfidf_vectorizer.transform(
        [profile["interests"]]
    )

    scores = cosine_similarity(
        profile_vector,
        tfidf_matrix
    ).flatten()

    ranked = courses.copy()
    ranked["score"] = scores

    return ranked.sort_values(
        "score",
        ascending=False
    ).head(top_k)

In [7]:
def rank_semantic(profile, top_k=10):
    profile_embedding = embedding_model.encode(
        [profile["interests"]]
    )

    scores = cosine_similarity(
        profile_embedding,
        course_embeddings
    ).flatten()

    ranked = courses.copy()
    ranked["score"] = scores

    return ranked.sort_values(
        "score",
        ascending=False
    ).head(top_k)

In [8]:
def rank_hybrid(profile, top_k=10):
    profile_embedding = embedding_model.encode(
        [profile["interests"]]
    )

    semantic_scores = cosine_similarity(
        profile_embedding,
        course_embeddings
    ).flatten()

    ranked = courses.copy()
    ranked["semantic_score"] = semantic_scores

    ranked["subject_fit"] = ranked["subject"].apply(
        lambda x: 1 if x in profile["preferred_subjects"] else 0
    )

    ranked["final_score"] = (
        0.70 * ranked["semantic_score"]
        + 0.30 * ranked["subject_fit"]
    )

    return ranked.sort_values(
        "final_score",
        ascending=False
    ).head(top_k)

In [9]:
profile = evaluation_profiles[0]

display(
    rank_tfidf(profile)[
        ["course", "title", "score"]
    ]
)

display(
    rank_semantic(profile)[
        ["course", "title", "score"]
    ]
)

display(
    rank_hybrid(profile)[
        ["course", "title", "final_score"]
    ]
)

,course,title,score
427,DATA C101,Data Engineering,0.348147
1566,PHYSICS 88,Data Science Applications in Physics,0.301712
447,DATA 188,Advanced Data Science Connector,0.298105
1015,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.286321
578,ENGIN 178,Statistics and Data Science for Engineers,0.278283
448,DATA 36,Data Scholars Seminar,0.263019
425,DATA C100,Principles & Techniques of Data Science,0.245281
1876,STAT 157,Seminar on Topics in Probability and Statistics,0.240771
173,ASTRON 128,Astronomy Data Science Laboratory,0.238910
453,DATA C102,"Data, Inference, and Decisions",0.216100


,course,title,score
578,ENGIN 178,Statistics and Data Science for Engineers,0.661259
412,COMPSCI 189,Introduction to Machine Learning,0.625625
456,DATA C140,Probability for Data Science,0.623366
455,DATA C131A,Statistical Methods for Data Science,0.616729
453,DATA C102,"Data, Inference, and Decisions",0.598894
1868,STAT 133,Concepts in Computing with Data,0.594744
1870,STAT 135,Concepts of Statistics,0.587110
425,DATA C100,Principles & Techniques of Data Science,0.582423
1874,STAT 154,Modern Statistical Prediction and Machine Lear...,0.582386
459,STAT C88S,Probability and Mathematical Statistics in Dat...,0.567416


,course,title,final_score
412,COMPSCI 189,Introduction to Machine Learning,0.737937
456,DATA C140,Probability for Data Science,0.736356
455,DATA C131A,Statistical Methods for Data Science,0.731710
453,DATA C102,"Data, Inference, and Decisions",0.719226
1868,STAT 133,Concepts in Computing with Data,0.716321
1870,STAT 135,Concepts of Statistics,0.710977
425,DATA C100,Principles & Techniques of Data Science,0.707696
1874,STAT 154,Modern Statistical Prediction and Machine Lear...,0.707670
459,STAT C88S,Probability and Mathematical Statistics in Dat...,0.697191
445,DATA 144,Data Mining and Analytics,0.688733


## 3. Ranking Metrics
The three ranking methods are evaluated using manually labeled relevant courses for each student profile. 

Two ranking metrics used: 

- **Recall@10** - how many relevant courses appear in the top 10 recommendations
- **NDCG@10** - how highly relevant courses are ranked within the top 10

In [10]:
def recall_at_k(ranked_courses, relevant_courses, k=10):
    top_k = set(ranked_courses["course"].head(k))
    relevant = set(relevant_courses)

    if len(relevant) == 0:
        return 0.0

    return len(top_k & relevant) / len(relevant)

In [11]:
import numpy as np

def ndcg_at_k(ranked_courses, relevant_courses, k=10):
    relevant = set(relevant_courses)

    gains = [
        1 if course in relevant else 0
        for course in ranked_courses["course"].head(k)
    ]

    dcg = sum(
        gain / np.log2(i + 2)
        for i, gain in enumerate(gains)
    )

    ideal_gains = [1] * min(len(relevant), k)

    idcg = sum(
        gain / np.log2(i + 2)
        for i, gain in enumerate(ideal_gains)
    )

    return dcg / idcg if idcg > 0 else 0.0

In [12]:
results = []

for profile in evaluation_profiles:
    tfidf_ranked = rank_tfidf(profile, top_k=10)
    semantic_ranked = rank_semantic(profile, top_k=10)
    hybrid_ranked = rank_hybrid(profile, top_k=10)

    for model_name, ranked in [
        ("TF-IDF", tfidf_ranked),
        ("Semantic", semantic_ranked),
        ("Hybrid", hybrid_ranked),
    ]:
        results.append({
            "profile": profile["name"],
            "model": model_name,
            "recall@10": recall_at_k(
                ranked,
                profile["relevant_courses"],
                k=10
            ),
            "ndcg@10": ndcg_at_k(
                ranked,
                profile["relevant_courses"],
                k=10
            )
        })

results_df = pd.DataFrame(results)
results_df

,profile,model,recall@10,ndcg@10
0,Machine Learning / Data Science,TF-IDF,0.333333,0.217795
1,Machine Learning / Data Science,Semantic,0.500000,0.399076
2,Machine Learning / Data Science,Hybrid,0.500000,0.528387
3,Product Analytics / PM,TF-IDF,0.666667,0.516506
4,Product Analytics / PM,Semantic,0.833333,0.892211
5,Product Analytics / PM,Hybrid,0.833333,0.892211
6,Economics / Public Policy,TF-IDF,0.333333,0.493523
7,Economics / Public Policy,Semantic,0.500000,0.610586
8,Economics / Public Policy,Hybrid,0.833333,0.823389
9,Healthcare / Computational Biology,TF-IDF,0.250000,0.151020


In [13]:
summary = results_df.groupby("model")[
    ["recall@10", "ndcg@10"]
].mean().sort_values(
    "ndcg@10",
    ascending=False
)

summary

,recall@10,ndcg@10
model,,
Hybrid,0.663810,0.606521
Semantic,0.516190,0.488840
TF-IDF,0.349286,0.316980


### Evaluation Observations
Using a fixed 0.70/0.30 semantic/subject split (not tuned on this data), the hybrid ranker achieves the strongest average performance across the evaluation profiles at 0.664 Recall@10 and 0.607 NDCG@10, ahead of semantic-only (0.516 / 0.489) and TF-IDF (0.349 / 0.317). Performance varies substantially by profile (e.g. Design / HCI has only a single labeled relevant course, so its score is close to binary), which is a sign the aggregate mean should be read with its per-profile spread, not on its own.

## 4. Error Analysis
Overall metrics favor the hybrid model but performance varies across student profiles. This section inspects profiles where semantic or hybrid ranking underperforms to identify failure modes. 

In [14]:
product_profile = next(
    p for p in evaluation_profiles
    if p["name"] == "Product Analytics / PM"
)

In [15]:
display(
    rank_tfidf(product_profile)[
        ["course", "title", "score"]
    ]
)

display(
    rank_semantic(product_profile)[
        ["course", "title", "score"]
    ]
)

display(
    rank_hybrid(product_profile)[
        ["course", "title", "final_score"]
    ]
)

,course,title,score
1963,UGBA 167,Special Topics in Marketing,0.303283
1961,UGBA 162,Brand Management and Strategy,0.294383
1934,UGBA 106,Marketing,0.285130
583,ENGIN 183D,Product Management,0.254907
1964,UGBA 168B,International Marketing,0.201864
1959,UGBA 160,Customer Insights,0.186135
1999,UGBA C5,Introduction to Entrepreneurship,0.179900
1547,PHYSICS 111B,Advanced Experimentation Laboratory,0.178144
66,ANTHRO 106,Primate Behavior,0.173058
1960,UGBA 161,Market Research: Tools and Techniques for Data...,0.164823


,course,title,score
583,ENGIN 183D,Product Management,0.593528
1960,UGBA 161,Market Research: Tools and Techniques for Data...,0.568208
1961,UGBA 162,Brand Management and Strategy,0.552303
1959,UGBA 160,Customer Insights,0.543921
1932,UGBA 104,Introduction to Business Analytics,0.499109
1997,UGBA 88,Data and Decisions,0.485294
1963,UGBA 167,Special Topics in Marketing,0.468910
1934,UGBA 106,Marketing,0.452799
1962,UGBA 162A,Product Branding and Branded Entertainment,0.446114
445,DATA 144,Data Mining and Analytics,0.441540


,course,title,final_score
583,ENGIN 183D,Product Management,0.715470
1960,UGBA 161,Market Research: Tools and Techniques for Data...,0.697745
1961,UGBA 162,Brand Management and Strategy,0.686612
1959,UGBA 160,Customer Insights,0.680745
1932,UGBA 104,Introduction to Business Analytics,0.649376
1997,UGBA 88,Data and Decisions,0.639705
1963,UGBA 167,Special Topics in Marketing,0.628237
1934,UGBA 106,Marketing,0.616959
1962,UGBA 162A,Product Branding and Branded Entertainment,0.612280
445,DATA 144,Data Mining and Analytics,0.609078


In [16]:
product_profile["relevant_courses"]

['UGBA 104', 'UGBA 160', 'UGBA 161', 'UGBA 162', 'INDENG 142A', 'ENGIN 183D']

In [17]:
results_df

,profile,model,recall@10,ndcg@10
0,Machine Learning / Data Science,TF-IDF,0.333333,0.217795
1,Machine Learning / Data Science,Semantic,0.500000,0.399076
2,Machine Learning / Data Science,Hybrid,0.500000,0.528387
3,Product Analytics / PM,TF-IDF,0.666667,0.516506
4,Product Analytics / PM,Semantic,0.833333,0.892211
5,Product Analytics / PM,Hybrid,0.833333,0.892211
6,Economics / Public Policy,TF-IDF,0.333333,0.493523
7,Economics / Public Policy,Semantic,0.500000,0.610586
8,Economics / Public Policy,Hybrid,0.833333,0.823389
9,Healthcare / Computational Biology,TF-IDF,0.250000,0.151020


In [18]:
summary

,recall@10,ndcg@10
model,,
Hybrid,0.663810,0.606521
Semantic,0.516190,0.488840
TF-IDF,0.349286,0.316980


### Evaluation Summary
The hybrid ranker outperforms semantic-only and TF-IDF baselines on this labeled set. However, the section below ("Weight Selection") replaces the earlier full-set grid search with leave-one-profile-out cross-validation, because tuning `semantic_weight` against the same 10 profiles used to report the score is data leakage — it lets the weight overfit to this specific label set rather than measuring how well it generalizes.

*This evaluation uses a small, hand-labeled set of 10 student profiles authored by the same person who built the ranker, so even the cross-validated numbers below should be read as a directional, controlled V1 benchmark — not a claim about real student outcomes.

## 5. Weight Selection (Leave-One-Profile-Out Cross-Validation)
The earlier approach (grid-searching `semantic_weight` against all 10 profiles, then reporting the score on those same 10 profiles) is data leakage: the weight is fit to the exact data it's evaluated on, so the resulting number doesn't say anything about how well that weight generalizes.

Instead, each profile below is scored using a weight chosen from the **other 9 profiles only** (leave-one-profile-out). The weight it never trained on is the one it's judged on, so this is the number worth trusting as a performance estimate.

In [19]:
def rank_hybrid_weighted(
    courses,
    interests,
    preferred_subjects,
    model,
    course_embeddings,
    semantic_weight=0.85
):
    profile_embedding = model.encode([interests])

    semantic_scores = cosine_similarity(
        profile_embedding,
        course_embeddings
    ).flatten()

    ranked = courses.copy()
    ranked["semantic_score"] = semantic_scores

    ranked["subject_fit"] = ranked["subject"].isin(
        preferred_subjects
    ).astype(int)

    subject_weight = 1 - semantic_weight

    ranked["final_score"] = (
        semantic_weight * ranked["semantic_score"]
        + subject_weight * ranked["subject_fit"]
    )

    return ranked.sort_values(
        "final_score",
        ascending=False
    )

In [20]:
weights = [
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
    1.00,
]

In [21]:
def mean_ndcg_for_weight(profiles, weight):
    scores = []

    for profile in profiles:
        ranked = rank_hybrid_weighted(
            courses,
            profile["interests"],
            profile["preferred_subjects"],
            embedding_model,
            course_embeddings,
            semantic_weight=weight
        )

        scores.append(
            ndcg_at_k(
                ranked,
                profile["relevant_courses"],
                10
            )
        )

    return sum(scores) / len(scores)


def select_best_weight(profiles, weights):
    """Pick the weight with the highest mean NDCG@10 over `profiles`."""

    return max(
        weights,
        key=lambda weight: mean_ndcg_for_weight(profiles, weight)
    )

In [22]:
loo_records = []

for i, held_out in enumerate(evaluation_profiles):
    train_profiles = (
        evaluation_profiles[:i] + evaluation_profiles[i + 1:]
    )

    # Weight is chosen using only the other 9 profiles, then scored
    # on the held-out profile it never saw.
    chosen_weight = select_best_weight(train_profiles, weights)

    ranked = rank_hybrid_weighted(
        courses,
        held_out["interests"],
        held_out["preferred_subjects"],
        embedding_model,
        course_embeddings,
        semantic_weight=chosen_weight
    )

    loo_records.append({
        "profile": held_out["name"],
        "chosen_weight": chosen_weight,
        "recall@10": recall_at_k(
            ranked, held_out["relevant_courses"], 10
        ),
        "ndcg@10": ndcg_at_k(
            ranked, held_out["relevant_courses"], 10
        ),
    })

loo_df = pd.DataFrame(loo_records)
loo_df

,profile,chosen_weight,recall@10,ndcg@10
0,Machine Learning / Data Science,0.6,0.500000,0.528387
1,Product Analytics / PM,0.6,0.833333,0.892211
2,Economics / Public Policy,0.6,0.833333,0.823389
3,Healthcare / Computational Biology,0.6,0.250000,0.246302
4,Engineering Systems / Optimization,0.6,0.400000,0.441258
5,Software / Systems,0.6,0.750000,0.831872
6,Statistics,0.6,0.571429,0.596941
7,Finance / Quantitative Analysis,0.6,0.500000,0.274171
8,Behavioral Science,0.6,1.000000,1.000000
9,Design / HCI,0.6,1.000000,0.430677


In [23]:
loo_summary = loo_df[["recall@10", "ndcg@10"]].agg(["mean", "std"])
loo_summary

,recall@10,ndcg@10
mean,0.663810,0.606521
std,0.256902,0.266438


The standard deviation next to each mean shows how much the cross-validated score swings across profiles — with only 10 profiles (some with as few as 1 labeled relevant course), a single profile can move the mean substantially, so the spread matters as much as the average.

Finally, for the weight actually shipped to the app, it's reasonable to fit against all 10 profiles (there's no more held-out data left to lose) — but its reported *held-out* performance should come from the LOOCV numbers above, not from scoring this weight on the same full set it was chosen from.

In [24]:
production_weight = select_best_weight(evaluation_profiles, weights)
production_weight

0.6

In [25]:
print("courses:", len(courses))
print("embeddings:", len(course_embeddings))

courses: 2077
embeddings: 2077
